In [19]:
import scanpy as sc
import numpy as np

# anndata = sc.read("./data/TBD_AnnData.h5ad")
anndata = sc.read("./data/RP-01KVA3R3JM7D71W683AYNXQH26.h5ad")

metadata = anndata.obs['louvain']
clusters = metadata.cat.categories.tolist()
n_cells = [len(metadata[metadata==cluster]) for cluster in clusters]

clusters = [cluster for cluster in zip(clusters, n_cells)]
clusters

[('B cells', 342),
 ('CD14+ Monocytes', 480),
 ('CD4 T cells', 1144),
 ('CD8 T cells', 316),
 ('Dendritic cells', 37),
 ('FCGR3A+ Monocytes', 150),
 ('Megakaryocytes', 15),
 ('Unassigned', 154)]

In [ ]:
obsm = anndata.obsm['X_umap']
clusters = metadata.cat.categories.tolist()
clusters

points = []

for cluster in clusters:
    cluster_mask = metadata == cluster
    points.append(
        obsm[cluster_mask].tolist()
    )
points[0]

In [64]:
gene_indices = [3930, 3931]
select_gene_idx=3931
metadata_field = "louvain"
split_by_field = "111"
split_by_type = "metadata"

obs = anndata.obs[metadata_field]
split_obs = anndata.obs[split_by_field]

def expression_formatter(
    include_zeros: bool,
    scale: bool,
    clip_range: Optional[List[float]],
    expression: np.ndarray,
    expression_scale: np.ndarray,
):
    n_expressed_cells = np.count_nonzero(expression)
    total_cells = len(expression)
    if total_cells == 0:
        return {
            "average_expression": 0,
            "coverage": 0,
            "n_expressed_cells": 0,
            "total_cells": 0,
        }
    coverage = n_expressed_cells / total_cells
    if n_expressed_cells > 0:
        average_expression = np.sum(expression_scale)
        average_expression /= total_cells if include_zeros else n_expressed_cells
    else:
        average_expression = 0
    return {
        "average_expression": average_expression,
        "coverage": coverage,
        "n_expressed_cells": n_expressed_cells,
        "total_cells": total_cells,
        "clip_range": clip_range,
        "include_zeros": include_zeros,
        "scale": scale,
    }

resp = {}
mtx = anndata.layers['log_normalize'].toarray()[:,gene_indices]

for i, gene_idx in enumerate(gene_indices):
    if gene_idx != select_gene_idx:
        continue
    gene_name = anndata.var_names[gene_idx]
    expression = mtx[:,i]
    if split_by_type == "gene":
        resp[gene_name] = {
            "data": [],
            "coverage_range": [],
            "expression_range": []
        }
    for split_idx, split_cluster in enumerate(list(split_obs.cat.categories)):
        if split_by_type == 'metadata':
            if split_cluster not in resp:
                resp[split_cluster] = {
                    "data": [],
                    "coverage_range": [],
                    "expression_range": []
                }
        split_mask = split_obs==split_cluster
        expression_split = expression[split_mask]
        x_meta = obs[split_mask]
        for cluster_idx, cluster in enumerate(list(x_meta.cat.categories)):
            cluster_mask = x_meta==cluster
            expression_cluster = expression_split[cluster_mask]
            if split_by_type=="gene":
                resp[gene_name]["data"].append({
                    "node":cluster,
                    "feature": split_cluster,
                    **expression_formatter(
                        include_zeros=False,
                        scale=False,
                        clip_range=[],
                        expression=expression_cluster,
                        expression_scale=expression_cluster

                    )
                })
            else:
                resp[split_cluster]["data"].append({
                    "node":cluster,
                    "feature": gene_name,
                    **expression_formatter(
                        include_zeros=False,
                        scale=False,
                        clip_range=[],
                        expression=expression_cluster,
                        expression_scale=expression_cluster

                    )
                })




resp



{'Cluster 1': {'data': [{'node': 'B cells',
    'feature': 'CD79B',
    'average_expression': 0,
    'coverage': np.float64(0.0),
    'n_expressed_cells': np.int64(0),
    'total_cells': 1,
    'clip_range': [],
    'include_zeros': False,
    'scale': False},
   {'node': 'CD14+ Monocytes',
    'feature': 'CD79B',
    'average_expression': 0,
    'coverage': 0,
    'n_expressed_cells': 0,
    'total_cells': 0},
   {'node': 'CD4 T cells',
    'feature': 'CD79B',
    'average_expression': np.float64(1.6687231258470185),
    'coverage': np.float64(0.0839041095890411),
    'n_expressed_cells': np.int64(49),
    'total_cells': 584,
    'clip_range': [],
    'include_zeros': False,
    'scale': False},
   {'node': 'CD8 T cells',
    'feature': 'CD79B',
    'average_expression': 0,
    'coverage': np.float64(0.0),
    'n_expressed_cells': np.int64(0),
    'total_cells': 9,
    'clip_range': [],
    'include_zeros': False,
    'scale': False},
   {'node': 'Dendritic cells',
    'feature': 'CD7

In [44]:
import pandas as pd

data =  [
			{
				"node": "B cells",
				"feature": "CD79A",
				"data": 0.38033294677734375
			},
			{
				"node": "CD14+ Monocytes",
				"feature": "CD79A",
				"data": -1.2051905393600464
			},
			{
				"node": "CD4 T cells",
				"feature": "CD79A",
				"data": -1.2016199827194214
			},
			{
				"node": "CD8 T cells",
				"feature": "CD79A",
				"data": -0.8922745585441589
			},
			{
				"node": "Dendritic cells",
				"feature": "CD79A",
				"data": -1.6146583557128906
			},
			{
				"node": "FCGR3A+ Monocytes",
				"feature": "CD79A",
				"data": -1.5694948434829712
			},
			{
				"node": "Megakaryocytes",
				"feature": "CD79A",
				"data": -2.340604543685913
			},
			{
				"node": "NK cells",
				"feature": "CD79A",
				"data": -1.2071713209152222
			},
			{
				"node": "B cells",
				"feature": "CD79B",
				"data": 0.7546921968460083
			},
			{
				"node": "CD14+ Monocytes",
				"feature": "CD79B",
				"data": -0.8941560983657837
			},
			{
				"node": "CD4 T cells",
				"feature": "CD79B",
				"data": -0.8132047057151794
			},
			{
				"node": "CD8 T cells",
				"feature": "CD79B",
				"data": -0.6043251752853394
			},
			{
				"node": "Dendritic cells",
				"feature": "CD79B",
				"data": -1.444570541381836
			},
			{
				"node": "FCGR3A+ Monocytes",
				"feature": "CD79B",
				"data": -0.6625850200653076
			},
			{
				"node": "Megakaryocytes",
				"feature": "CD79B",
				"data": -1.0729281902313232
			},
			{
				"node": "NK cells",
				"feature": "CD79B",
				"data": -0.25997015833854675
			}
		]



data

df = pd.DataFrame(data)
exp=df.groupby(by="feature")["data"].get_group("CD79A").to_numpy()
mask = np.ones(len(exp), dtype=bool)
mask[[0,1]]=False
arr = [[1,2,3], [3,4,5]]
np.array(arr).flatten().min()
np.array(arr).flatten().max()

# mask




# df



# df




np.int64(5)

In [ ]:
from pydantic import BaseModel
from typing import Optional, List, Literal, Union

class TextStyle(BaseModel):
    color: Optional[str] = "#565657"
    fontStyle: Optional[Literal["normal","bold"]] = "normal"
    fontWeight: Optional[Literal["normal","bold"]] = "normal"
    fontSize: Optional[int] = 12
    fontFamily: Optional[str] =  "Roboto, sans-serif"

class Grid(BaseModel):
    show: Optional[bool] = False
    left: Optional[float] = 40
    top: Optional[float] = 20
    width: Optional[float] = 0
    height: Optional[float] = 0
    contentLabel: Optional[bool] = True
    bottom: Optional[float] = 40
    right: Optional[float] = 20

class BiovinciOption(BaseModel):
    textStyle: TextStyle
    grid: List[Grid]

class AxisLabel(BaseModel):
    interval: Optional[int] = 0
    rotate: Optional[float] = -45
    width: Optional[float] = 0
    interval: Optional[int] = 0
    interval: Optional[int] = 0

class XAxis(BaseModel):
    type: Optional[str] = 'category'
    data: Optional[List[Union[str, float, int]]] = None
    name: Optional[str] = None
    position: Optional[Literal["top", "bottom"]] = "top"
    axisLabel: 



option = BiovinciOption(
    textStyle=TextStyle(),
    grid=[Grid(
        width=100,
        height=200
    )],

)
option.model_dump()


{'textStyle': {'color': '#565657',
  'fontStyle': 'normal',
  'fontWeight': 'normal',
  'fontSize': 12,
  'fontFamily': 'Roboto, sans-serif'},
 'grid': {'show': False,
  'left': 40,
  'top': 20,
  'width': 100.0,
  'height': 200.0,
  'contentLabel': True,
  'bottom': 40,
  'right': 20}}